In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./final_data.csv')

In [ ]:

# 1. 위치 정보(주)가 존재하는 데이터만 추출 (기준점)
known_state_data = df.dropna(subset=['MERCHANT_STATE'])

# 2. 거래 설명 문구별로 '주(State)'가 몇 종류인지 확인
state_mapping = known_state_data[['TRANSACTION_DESCRIPTION', 'MERCHANT_STATE']].drop_duplicates()

# 3. 설명 문구당 주(State)가 '딱 1개'인 '주-청정 문구' 리스트 추출
state_v_counts = state_mapping['TRANSACTION_DESCRIPTION'].value_counts()
clean_state_desc_list = state_v_counts[state_v_counts == 1].index

# 4. [문구 : 주] 매핑 사전 제작
state_lookup_map = state_mapping[state_mapping['TRANSACTION_DESCRIPTION'].isin(clean_state_desc_list)].set_index('TRANSACTION_DESCRIPTION')['MERCHANT_STATE']

# 5. 시뮬레이션: MERCHANT_STATE가 결측치인 행 중 복구 가능한 건수 계산
nan_state_mask = df['MERCHANT_STATE'].isna()
fixable_state_mask = nan_state_mask & df['TRANSACTION_DESCRIPTION'].isin(clean_state_desc_list)

total_nan_state = nan_state_mask.sum()
fixable_state_count = fixable_state_mask.sum()

# 6. 복구 가능한 가맹점 이름 추출 (중복 제거 후 상위 10개 예시)
fixable_merchant_names = df.loc[fixable_state_mask, 'MERCHANT_NAME'].unique()
fixable_names_list = list(fixable_merchant_names)

# 7. 결과 출력
print(f"📊 [가맹점 주(State) 기준 복구 시뮬레이션]")
print(f"- 현재 MERCHANT_STATE 결측치: {total_nan_state:,}건")
print(f"- 💎 주(State)를 100% 확신하여 복구 가능한 건수: {fixable_state_count:,}건")
print(f"- 📈 주(State) 복구 예상 성공률: {(fixable_state_count/total_nan_state):.2%}")
print("-" * 45)

if fixable_state_count > 0:
    print(f"✅ 복구 가능한 가맹점 예시 (총 {len(fixable_names_list):,}개 가맹점):")
    # 너무 많을 수 있으므로 상위 10개만 출력
    for name in fixable_names_list[:10]:
        print(f"  • {name}")
    
    if len(fixable_names_list) > 10:
        print(f"  ...외 {len(fixable_names_list) - 10}개의 가맹점이 더 존재합니다.")
else:
    print("❌ 복구 가능한 가맹점이 없습니다.")

print("-" * 45)
print(f"💡 '도시'까지 따졌을 때보다 훨씬 많은 결측치를 '주' 단위에서 살려낼 수 있습니다.")

📊 [가맹점 주(State) 기준 복구 시뮬레이션]
- 현재 MERCHANT_STATE 결측치: 574,145건
- 💎 주(State)를 100% 확신하여 복구 가능한 건수: 281,140건
- 📈 주(State) 복구 예상 성공률: 48.97%
---------------------------------------------
✅ 복구 가능한 가맹점 예시 (총 279개 가맹점):
  • BURGER KING
  • STARBUCKS
  • MCDONALDS
  • USA CANTEEN VENDING
  • WENDYS
  • CHICK-FIL-A
  • LITTLE CAESARS
  • SONIC DRIVE-IN
  • TACO BELL
  • CHIPOTLE
  ...외 269개의 가맹점이 더 존재합니다.
---------------------------------------------
💡 '도시'까지 따졌을 때보다 훨씬 많은 결측치를 '주' 단위에서 살려낼 수 있습니다.


In [11]:
df.columns

Index(['ACCOUNT_ID', 'CARD_ID', 'TRANSACTION_ID', 'GROSS_TRANSACTION_AMOUNT',
       'TRANSACTION_DATE', 'TRANSACTION_TYPE', 'TRANSACTION_STATE',
       'TRANSACTION_CITY', 'MERCHANT_STATE', 'MERCHANT_CITY',
       'MERCHANT_CATEGORY_LEVEL_1', 'MERCHANT_CATEGORY_LEVEL_2',
       'MERCHANT_CATEGORY_LEVEL_3', 'TRANSACTION_DESCRIPTION', 'CARD_TYPE',
       'CARD_HOLDER_STATE', 'CARD_HOLDER_CITY', 'CARD_HOLDER_GENERATION',
       'CARD_HOLDER_TOTAL_TRANSACTION_COUNT', 'CARD_HOLDER_TOTAL_SPEND',
       'CARD_HOLDER_AVERAGE_LTM_SPEND',
       'CARD_HOLDER_AVERAGE_LTM_TRANSACTION_COUNT', 'CARD_HOLDER_VINTAGE',
       'CARD_PRESENT_INDICATOR', 'MERCHANT_ID', 'MERCHANT_NAME',
       'SHOPPER_CLASSIFICATION'],
      dtype='object')

In [13]:
# 6. 복구 대상 데이터 추출 (가맹점명과 매핑된 주 정보 결합)
# 복구 가능한 행들만 필터링
recovery_sample = df[fixable_state_mask][['MERCHANT_NAME', 'TRANSACTION_DESCRIPTION']].copy()

# 미리 만들어둔 state_lookup_map을 이용해 복구될 '주' 값을 매칭
recovery_sample['RESTORED_STATE'] = recovery_sample['TRANSACTION_DESCRIPTION'].map(state_lookup_map)

# 중복 제거 (어떤 가맹점이 어떤 주로 복구되는지 '쌍'만 보기 위함)
recovery_result = recovery_sample[['MERCHANT_NAME', 'RESTORED_STATE', 'TRANSACTION_DESCRIPTION']].drop_duplicates()

# 7. 결과 출력
print(f"📊 [복구 대상 가맹점 상세 내역]")
print(f"- 총 {len(recovery_result):,}개의 고유 가맹점-주 매칭 발견\n")

# 상위 20개 출력 (데이터가 많을 수 있으므로)
print(recovery_result.head(20).to_string(index=False))

if len(recovery_result) > 20:
    print(f"\n...외 {len(recovery_result) - 20}건의 매칭이 더 존재합니다.")

📊 [복구 대상 가맹점 상세 내역]
- 총 47,161개의 고유 가맹점-주 매칭 발견

      MERCHANT_NAME RESTORED_STATE                  TRANSACTION_DESCRIPTION
        BURGER KING             FL                          BURGER KING #91
          STARBUCKS             WA                   STARBUCKS 800-782-7282
          MCDONALDS             FL                         MCDONALD'S M2103
USA CANTEEN VENDING             FL CANTEEN VENDING        GAINESVILLE  FLUS
USA CANTEEN VENDING             FL CANTEEN VENDING        JACKSONVILLE FLUS
          MCDONALDS             MI                     \"MCDONALD'S F5568\"
          MCDONALDS             OR                        MCDONALD'S F34916
          MCDONALDS             AL                         MCDONALD'S F3158
             WENDYS             TX                              WENDYS 9621
        CHICK-FIL-A             NC                       CHICK-FIL-A #04772
             WENDYS             MI                             WENDY'S 8338
             WENDYS             MI     

In [12]:
df[['MERCHANT_NAME', 'MERCHANT_STATE', 'TRANSACTION_DESCRIPTION']]

,MERCHANT_NAME,MERCHANT_STATE,TRANSACTION_DESCRIPTION
0,WENDYS,OH,WENDY'S 659 WESTLAKE OHUS
1,JIMMY JOHNS GOURMET SANDWICHES,MI,JIMMY JOHNS - 0689
2,SUBWAY,OH,Subway 52906
3,MCDONALDS,IN,McDonalds 30963
4,CULVERS,MN,CULVERS OF DELANO
...,...,...,...
4771535,PANDA EXPRESS,TX,PANDA EXPRESS #3455 KATY TXUS
4771536,MCDONALDS,OH,McDonalds 31071
4771537,USCONNECT,TN,USCONNECT VANVN VEND N
4771538,KFC,NaN,Dominos Kaiserslauter


In [9]:
fixable_names_list

['BURGER KING',
 'STARBUCKS',
 'MCDONALDS',
 'USA CANTEEN VENDING',
 'WENDYS',
 'CHICK-FIL-A',
 'LITTLE CAESARS',
 'SONIC DRIVE-IN',
 'TACO BELL',
 'CHIPOTLE',
 'DOMINOS',
 'FUZZYS TACO',
 'FIVE GUYS BURGERS AND FRIES',
 'PANDA EXPRESS',
 'NO ENTITY',
 'DAIRY QUEEN',
 'CRANE MERCHANDISING SYSTEMS',
 'DUNKIN DONUTS',
 'POPEYES LOUISIANA KITCHEN',
 'WHATABURGER',
 'HUNGRY HOWIES PIZZA',
 '365 RETAIL MARKETS',
 'DUTCH BROS. COFFEE',
 'COOK OUT',
 'TIM HORTONS',
 'WHITE CASTLE',
 'WINGSTOP',
 'BASKIN ROBBINS',
 'NAYAX',
 'HARDEES',
 'BOJANGLES FAMOUS CHICKEN N BISCUITS',
 'CULVERS',
 'ZAXBYS',
 'COCA COLA',
 'ARBYS',
 'DEL TACO',
 'SUBWAY',
 'EINSTEIN NOAH RESTAURANT',
 'JACK IN THE BOX',
 'PAPA JOHNS',
 'JIMMY JOHNS GOURMET SANDWICHES',
 'DONATOS PIZZA',
 'SHIPLEY DO-NUTS',
 'CHICKEN EXPRESS',
 'RAISING CANES CHICKEN FINGERS',
 'ARAMARK',
 'DUCK DONUTS',
 '3 SQUARE MARKET',
 'SARKU JAPAN',
 'MOES SOUTHWEST GRILL',
 'FREDDYS FROZEN CUSTARD',
 'USCONNECT',
 'SLIM CHICKENS',
 'KRISPY KREME',

In [5]:
import pandas as pd

# 1. 위치 정보(주)와 가맹점 이름이 존재하는 데이터만 추출 (기준점)
# 이름과 주 정보가 모두 있는 데이터를 기준으로 학습합니다.
known_data = df.dropna(subset=['MERCHANT_STATE', 'MERCHANT_NAME'])

# 2. 거래 설명 문구별로 '주(State)'와 '이름(Name)'이 몇 종류인지 확인
# 설명 문구 하나에 이름이나 주가 여러 개 섞여 있는 경우를 걸러내기 위함입니다.
mapping_base = known_data[['TRANSACTION_DESCRIPTION', 'MERCHANT_STATE', 'MERCHANT_NAME']].drop_duplicates()

# 3. 설명 문구당 [주]와 [이름]이 '딱 1개'씩만 존재하는 '청정 문구' 리스트 추출
counts = mapping_base['TRANSACTION_DESCRIPTION'].value_counts()
clean_desc_list = counts[counts == 1].index

# 4. [문구 : 주] 및 [문구 : 이름] 매핑 사전 제작
# clean_desc_list에 해당하는 데이터만 남겨서 사전으로 만듭니다.
lookup_df = mapping_base[mapping_base['TRANSACTION_DESCRIPTION'].isin(clean_desc_list)].set_index('TRANSACTION_DESCRIPTION')
state_lookup_map = lookup_df['MERCHANT_STATE']
name_lookup_map = lookup_df['MERCHANT_NAME']

# 5. 시뮬레이션 및 데이터 매핑
# 원본 데이터 복사본 생성
df_sim = df.copy()

# 결측치 여부 확인
nan_mask = df_sim['MERCHANT_STATE'].isna() & df_sim['MERCHANT_NAME'].isna()
fixable_mask = df_sim['TRANSACTION_DESCRIPTION'].isin(clean_desc_list) & nan_mask

# 실제 복구 수행 (새로운 컬럼에 할당해보기)
df_sim.loc[fixable_mask, 'RECOVERED_STATE'] = df_sim.loc[fixable_mask, 'TRANSACTION_DESCRIPTION'].map(state_lookup_map)
df_sim.loc[fixable_mask, 'RECOVERED_NAME'] = df_sim.loc[fixable_mask, 'TRANSACTION_DESCRIPTION'].map(name_lookup_map)

# 6. 결과 출력
total_nan = nan_mask.sum()
fixable_count = fixable_mask.sum()

print(f"📊 [가맹점 정보(Name/State) 통합 복구 시뮬레이션]")
print(f"- 현재 주요 결측치(Name & State 모두 없는 경우): {total_nan:,}건")
print(f"- 💎 확신하여 복구 가능한 건수: {fixable_count:,}건")
print(f"- 📈 주(State) 복구 예상 성공률: {(fixable_count/total_nan if total_nan > 0 else 0):.2%}")
print("-" * 45)

# 7. 복구된 샘플 출력 (어떤 이름으로 채워졌는지 확인)
if fixable_count > 0:
    print("✅ [복구 샘플 (상위 5건)]")
    sample = df_sim[fixable_mask][['TRANSACTION_DESCRIPTION', 'RECOVERED_NAME', 'RECOVERED_STATE']].head(5)
    print(sample)

📊 [가맹점 정보(Name/State) 통합 복구 시뮬레이션]
- 현재 주요 결측치(Name & State 모두 없는 경우): 0건
- 💎 확신하여 복구 가능한 건수: 0건
- 📈 주(State) 복구 예상 성공률: 0.00%
---------------------------------------------
